# b04 왼쪽 견봉을 직접 찍고 체간 기울기 비교하기
**모든 셀 실행(Run All Cells)** 후 아래 두 탭에서 **선수 본인의 왼쪽 견봉**을 한 번씩 찍습니다. 9번은 이륙 후보, 21번은 무릎 통과입니다. 어깨 바깥쪽에서 견봉으로 판단하는 끝점을 두 장 모두 같은 기준으로 선택하세요. 원본 크기로 확대할 수 있습니다.

클릭할 때마다 자동 저장됩니다. 옷 때문에 위치를 추정했다면 먼저 **찍을 점 → 옷에 가려 추정**을 선택하고 클릭하세요. 점을 정하기 어려우면 **가려서 판독 불가** 또는 **위치 불확실**을 누르세요. 두 장을 마치고 **체간 변화 비교**를 누르면 처음으로 소켓과 모델 예측을 겹쳐 봅니다.

### 1. 두 장의 원본과 저장 위치 확인

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.evaluation.acromion import MANUAL, MODEL, acromion_reads, compare_trunk, plot_lines
from wlcv.evaluation.hip_fold import FRAMES, VIDEO

info = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").loc[VIDEO]
split = pd.read_csv(SPLIT).set_index("video_id").at[VIDEO, "role"]
assert (info.person_id, split, info.near_side) == ("b04", "train", "left")
video_path = VIDEOS / info.filename
shoulder_folder = ANNOTATIONS / "train_acromion_review"
socket_folder = ANNOTATIONS / "train_hip_review"
output = ROOT / "outputs/pose_baseline/b04_acromion_trunk"

### 2. 견봉을 찍고 결과 보기
노란 선은 **수동 견봉과 같은 프레임의 소켓 1·2차 추정 중점**, 청록색 선은 **YOLO26n-pose의 왼쪽 어깨와 왼쪽 골반**을 잇습니다. 체간 기울기는 화면 수직선에서 벗어난 각도입니다. **변화량 = 21번 각도 − 9번 각도**이므로 음수는 더 세워진 방향, 양수는 더 기울어진 방향입니다.

소켓 추정을 포함하므로 수동 각도는 참고값입니다. 판독 불가나 모델 좌표 누락(관절 신뢰도 0.50 미만 포함)은 빈칸으로 남깁니다. 다시 열면 저장한 견봉점을 볼 수 있습니다. 점을 수정하면 **체간 변화 비교**를 다시 누르세요.

In [ ]:
tabs = widgets.Tab(children=[
    pose_point_editor(video_path, frame, info.person_id, info.near_side, split=split,
                      annotations=shoulder_folder, joints=("어깨",), trunk_review=True,
                      frame_label=f"b04 {frame}번 · 왼쪽 견봉")
    for frame in FRAMES
])
tabs.set_title(0, "9번 · 이륙 후보")
tabs.set_title(1, "21번 · 무릎 통과")

button = widgets.Button(description="체간 변화 비교")
report = widgets.Output()

def show_comparison(_):
    report.clear_output(wait=True)
    with report:
        pending = [f for f, p in acromion_reads(shoulder_folder).items() if p is None]
        if pending:
            print(f"{pending}번의 왼쪽 견봉 위치 또는 판독 불가 상태를 먼저 저장하세요.")
            return
        weights = ROOT / "models/yolo26n-pose.pt"
        if not weights.is_file():
            print(f"로컬 가중치를 확인하세요: {weights}")
            return
        points = compare_trunk(video_path, YOLO(str(weights)), shoulder_folder, socket_folder)
        angles = points.pivot(index="기준", columns="프레임", values="각도 (°)").reindex([MANUAL, MODEL])
        angles["변화 (°)"] = angles[21] - angles[9]
        display(angles.round(2))
        display(points[["프레임", "기준", "어깨 판독", "골반 판독"]])
        delta = angles["변화 (°)"]
        if delta.notna().all():
            print(f"변화량 차이 (수동 − YOLO): {delta[MANUAL] - delta[MODEL]:+.2f}°")
        else:
            print("좌표가 부족하거나 두 점이 겹쳐 변화량 차이를 계산할 수 없습니다.")
        fig = plot_lines(video_path, points)
        output.mkdir(parents=True, exist_ok=True)
        points.to_csv(output / "frames.csv", index=False)
        angles.to_csv(output / "angle_changes.csv")
        fig.savefig(output / "trunk_comparison.png", dpi=140, bbox_inches="tight")
        plt.show()

button.on_click(show_comparison)
display(widgets.VBox([tabs, button, report]))